In [1]:
import json

JSON_PATH = 'jmdict-eng-3.6.2.json'

def load_dictionary(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # JMDict-simplified usually has a root key called 'words'
        return data.get('words', [])
    
JISHO = load_dictionary(JSON_PATH)
print(len(JISHO))

import pykakasi
import re
import traceback

def get_html_furigana(text):
    kks = pykakasi.kakasi()
    result = kks.convert(text)
    
    html_output = ""
    
    for item in result:
        kanji_block = item['orig']
        reading = item['hira']
        
        if kanji_block == reading:
            html_output += kanji_block
            continue

        # Logic to handle Okurigana (like the 'き' in '生きる')
        # We find where the trailing hiragana starts
        match = re.search(r'([ぁ-ん]+)$', kanji_block)
        
        if match:
            okurigana = match.group(1)
            # Remove okurigana from the end of both the kanji block and the reading
            base_kanji = kanji_block[:-len(okurigana)]
            base_reading = reading[:-len(okurigana)]
            
            html_output += f"<ruby>{base_kanji}<rt>{base_reading}</rt></ruby>{okurigana}"
        else:
            # No okurigana found (pure kanji word like 日本語)
            html_output += f"<ruby>{kanji_block}<rt>{reading}</rt></ruby>"
            
    return html_output


214926


In [2]:
import requests

POS = {"n" : "Noun", "adj-i": "い-Adjective", "adj-na": "な-Adjective", "int" : "Interjection", "v1" : "Ichidan Verb", "v5m" : "Godan Verb", "v5g" : "Godan Verb", "v5k" : "Godan Verb", "v5k-s" : "Godan Verb", "v5t" : "Godan Verb", "v5b" : "Godan Verb", "v5u" : "Godan Verb", "v5r" : "Godan Verb", "v5s" : "Godan Verb", "vs-i" : "Suru Verb", "conj":"Conjuction", "vt": "Transitive Verb", "vi" : "Intransitive Verb", "adv" : "Adverb", "exp" : "Expression", "vs" : "Suru Verb", "num" : "Number", "pn" : "Pronoun", "suf" : "Suffix", "aux-v" : "Auxiliary Verb", "adj-no" : "Adjective Noun", "adj-pn": "Pre-noun Adjective", "v5r-i": "Godan Verb", "ctr" : "Counter", "n-suf" : "Noun Suffix", "n-pref" : "Noun Prefix", "pref" : "Prefix", "adv-to" : "と-Adverb", "adj-f": "Verb acting Prenominally", "vk" : "Kuru Verb", "v5n" : "Godan Verb", "vn" : "Godan Verb", "prt" : "Particle", "aux-adj" : "Auxiliary Adjective", "adj-ix" : "い-Adjective"}

import requests
import urllib.parse

def word_search(kanji_query, reading_query, similar_limit=3):
    exact_results = []
    similar_results = []
    
    k_queries = [k.strip() for k in kanji_query.split(",")] if kanji_query else []
    r_queries = [r.strip() for r in reading_query.split(",")] if reading_query else []
    
    # # --- STEP 1: LOCAL SEARCH ---
    # for entry in JISHO:
    #     score = 0
    #     is_exact = False
    #     
    #     kanji_elements = entry.get('kanji', [])
    #     kana_elements = entry.get('kana', [])
    #     kanji_list = [k.get('text') for k in kanji_elements]
    #     kana_list = [r.get('text') for r in kana_elements]
    #     is_common = any(k.get('common') for k in kanji_elements) or any(r.get('common') for r in kana_elements)
    # 
    #     k_exact = any(q in kanji_list for q in k_queries)
    #     r_exact = any(q in kana_list for q in r_queries)
    #     
    #     if k_exact or r_exact:
    #         is_exact = True
    #         score += 100
    #         if k_exact and r_exact: score += 50 
    #     elif not is_exact:
    #         match_found = False
    #         if k_queries:
    #             for q in k_queries:
    #                 for k_text in kanji_list:
    #                     if q in k_text:
    #                         score += 50 if k_text.startswith(q) else 20
    #                         match_found = True
    #                 if not match_found and not kanji_list:
    #                     for r_text in kana_list:
    #                         if q in r_text:
    #                             score += 30 if r_text.startswith(q) else 10
    #                             match_found = True
    #         elif r_queries:
    #             for q in r_queries:
    #                 for r_text in kana_list:
    #                     if q in r_text:
    #                         score += 50 if r_text.startswith(q) else 20
    #                         match_found = True
    # 
    #     if score > 0:
    #         item = {
    #             "japanese": [{"word": k.get('text'), "reading": kana_list[0] if kana_list else ""} for k in kanji_elements],
    #             "senses": [{"parts_of_speech": s.get('partOfSpeech', []), "english_definitions": s.get('gloss', [])} for s in entry.get('sense', [])],
    #             "score": score + (5 if is_common else 0)
    #         }
    #         if not item["japanese"] and kana_list:
    #             item["japanese"].append({"word": kana_list[0], "reading": ""})
    #         
    #         if is_exact: exact_results.append(item)
    #         else: similar_results.append(item)

    # --- STEP 2: FALLBACK TO ONLINE API ---
    if True:
        print(f"Fallback to API: {kanji}")
        # We use the Kanji query as primary search, fallback to reading
        search_term = k_queries[0] if k_queries else (r_queries[0] if r_queries else "")
        if search_term:
            encoded_query = urllib.parse.quote(search_term)
            api_url = f"https://jisho.org/api/v1/search/words?keyword={encoded_query}"
            
            try:
                response = requests.get(api_url, timeout=5)
                response.raise_for_status()
                api_data = response.json().get('data', [])
                
                for api_entry in api_data:
                    # Translate Jisho API format to your Local format
                    translated_item = {
                        "japanese": [
                            {"word": j.get('word', j.get('reading')), "reading": j.get('reading', "")} 
                            for j in api_entry.get('japanese', [])
                        ],
                        "senses": [
                            {"parts_of_speech": s.get('parts_of_speech', []), "english_definitions": [
                                {'lang': 'eng', 'gender': None, 'type': None, 'text': e} for e in s.get('english_definitions', [])
                            ]}
                            for s in api_entry.get('senses', [])
                        ]
                    }
                    # Jisho API results are already ordered by relevance
                    exact_results.append(translated_item)
            except Exception as e:
                print(f"Online API Error for {search_term}: {e}")

    # --- STEP 3: DEDUPLICATION & FINAL OUTPUT ---
    def get_id(res): return res["japanese"][0].get("word", "") + res["japanese"][0].get("reading", "")
    
    unique_exact = []
    seen = set()
    for r in sorted(exact_results, key=lambda x: x.get('score', 0), reverse=True):
        uid = get_id(r)
        if uid not in seen:
            unique_exact.append(r)
            seen.add(uid)

    unique_similar = []
    for r in sorted(similar_results, key=lambda x: x.get('score', 0), reverse=True):
        uid = get_id(r)
        if uid not in seen:
            unique_similar.append(r)
            seen.add(uid)

    # Cleanup scores and return
    for r in unique_exact + unique_similar: r.pop('score', None)
    return {"data": unique_exact}, {"data": unique_similar[:similar_limit]}

def GetSentences(query: str, reverse : bool = False):
    
    base = f"https://tatoeba.org/en/api_v0/search?from=jpn&has_audio=&list=&native=&original=&orphans=&query={query}&sort=relevance&sort_reverse=&tags=&to=eng&trans_filter=limit&trans_has_audio=&trans_link=&trans_native=&trans_orphan=&trans_to=eng&trans_unapproved=&trans_user=&unapproved=any&user=&word_count_max=50&word_count_min=3"
    
    try:
        res = requests.get(base)
        res.raise_for_status()
        data = res.json()
    except requests.RequestException as e:
        print(f"Error calling Tatoeba API: {e}")
        return
    
    sentences = data.get("results", {})
    
    if len(sentences) == 0:
        if reverse is False:
            return GetSentences(query, True)
        return "",""

    try:
        if reverse is True:
            return sentences[0]["translations"][0][0]["text"], sentences[0]["text"]
        return sentences[0]["text"], sentences[0]["translations"][0][0]["text"]
    except:
        if reverse is False:
            return GetSentences(query, True)
        return "",""
  
import csv
import os
  
def append_to_csv(file_name, data):

    # Check if the file exists to determine if we need a header (optional)
    file_exists = os.path.isfile(file_name)
    data =  [d.replace(";", ",") if type(d) is str else d for d in data]
    # Opening in 'a' (append) mode. 'newline=""' prevents blank rows on Windows.
    with open(file_name, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file, delimiter=';')
        writer.writerow(data)
        

In [4]:
def word_construct(kanji, meaning, reading, recurse=True):
    
    if kanji == "": kanji = reading
    elif reading == "": reading = kanji
    
    #print(f"Hello {kanji} + {reading} + {meaning}")
    main_pos = ""
    extra_data, reference_data = word_search(kanji.replace("～",""), reading.replace("～",""))
    
    meanings = []
    maxmeanings = 3
    
    try:
        
        if len(extra_data) > 0:
            for i, data in enumerate(extra_data["data"][0]["senses"]):
                if i >= maxmeanings:
                    break
                if main_pos == "": main_pos = ", ".join(data["parts_of_speech"])
                if not recurse: break
                mean = "; ".join(m["text"] for m in data["english_definitions"])
                meanings.append((mean, ", ".join(data["parts_of_speech"])))
    except:
        print(extra_data)
        raise Exception("")
      
    jp, en = "", ""
    kanji_ref_link = []
    reference_Words = []
    if recurse:      
        jp, en = GetSentences(kanji.replace("～",""))
        
        jp = get_html_furigana(jp)
    
        for ref in reference_data["data"]:
            kanji_ref = ref["japanese"][0]["word"]
            
            hiragana_ref = ref["japanese"][0]["reading"]
            meaning_ref = ref["senses"][0]["english_definitions"][0]["text"]
            reference_Words.append(word_construct(kanji_ref, meaning_ref, hiragana_ref, False)[0])
    
        kanji_ref_link = [r[0] for r in reference_Words]
    
    while len(kanji_ref_link) < 3:
        kanji_ref_link.append("")
        
    return [kanji, reading, meaning, main_pos, 5 if recurse else 0, en, jp,
            meanings[0][0] if len(meanings)>0 else "", meanings[0][1] if len(meanings)>0 else "", 
            meanings[1][0] if len(meanings)>1 else "", meanings[1][1] if len(meanings)>1 else "", 
            meanings[2][0] if len(meanings)>2 else "", meanings[2][1] if len(meanings)>2 else ""] + kanji_ref_link, reference_Words

#[kanji, reading, meaning, pos, level, en_sent, jp_sent, om1, opos1, om2, opos2, om3, opos3, reference]
            
#print(word_search("キログラム", "キログラム")[0])
#コピーする;コピーする;to copy
#print(word_construct("ラジオカセ", "radio cassette player", "ラジオカセ"))


In [5]:
import csv
import traceback
from tqdm import tqdm # Import the progress bar

# Path to your file
file_path = 'JLPT-N5.csv'
from tqdm import tqdm

lastKan = ""
lastdata = ""



# ... your other functions like word_construct and append_to_csv ...

try:
    # 1. Count total rows first to set the bar length
    with open(file_path, mode='r', encoding='utf-8') as f:
        total_rows = sum(1 for line in f)

    with open(file_path, mode='r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter=';')

        # 2. Wrap the reader in tqdm
        # desc: The label shown next to the bar
        # total: The total number of iterations
        for row in tqdm(reader, total=total_rows, desc="Processing Vocab"):

            if len(row) >= 3:
                kanji = row[0].strip()
                lastKan = kanji
                reading = row[1].strip()
                meaning = row[2].strip()
                    
                # Call your construction logic
                main, ref = word_construct(kanji, meaning, reading)
                
                append_to_csv("output2.csv", main)
                for r in ref:
                    append_to_csv("output2.csv", r)

            else:
                # Use tqdm.write instead of print to avoid breaking the bar
                tqdm.write(f"Row skip/Short row: {row}")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e} + {lastKan}")
    traceback.print_exc()

Processing Vocab:   0%|          | 0/718 [00:00<?, ?it/s]

Fallback to API: ﻿ああ


Processing Vocab:   0%|          | 1/718 [00:02<33:59,  2.84s/it]

Fallback to API: 会う


Processing Vocab:   0%|          | 2/718 [00:04<23:10,  1.94s/it]

...

Fallback to API: 朝


Processing Vocab:   2%|▏         | 12/718 [00:18<16:01,  1.36s/it]

Fallback to API: 朝御飯


Processing Vocab:   2%|▏         | 12/718 [00:19<19:36,  1.67s/it]


KeyboardInterrupt: 